# 기업 활동 기반 드론 배송 B2B 수요 거점 분석

**목적:** 성남시 내 법인 기업의 현재 밀집 현황(기업3)과 최근 1년간 신설 법인 증가 추이(기업4)를 결합하여,  
드론 배송 B2B 수요가 높거나 향후 성장 가능성이 있는 행정동을 식별하고 버티포트 후보 구역을 제안합니다.

| 단계 | 내용 |
|------|------|
| 1단계 | 데이터 전처리 — 기업3(단일 시점) + 기업4(13개월) |
| 2단계 | 지표 산출 — 법인 밀도, 드론 친화 업종 비율, 신설 추이, 상/하반기 가속도 |
| 3단계 | 기업 활동 활성화 지수 산출 (가중 합산) |
| 4단계 | 시각화 — 단계구분도 + 월별 추이 + 지수 순위 |

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import folium
from folium import Choropleth
import warnings
import glob
import os
import json

warnings.filterwarnings('ignore')

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

print('라이브러리 로드 완료')

In [ ]:
# 경로 설정 (이 노트북은 02_analysis/ 에 위치)
NB_DIR      = os.path.dirname(os.path.abspath('__file__'))
BASE_DIR    = os.path.join(NB_DIR, '..')
DATA_DIR    = os.path.join(BASE_DIR, '..', 'Data')
PROC_DIR    = os.path.join(BASE_DIR, 'processed')
OUT_DIR     = os.path.join(PROC_DIR)

CORP3_PATH  = os.path.join(DATA_DIR, '기업3', '기업3_법인기업_202512_성남시.csv')
CORP4_GLOB  = os.path.join(DATA_DIR, '기업4_신규기업', 'gg_corp4_new_*_성남시.csv')
BOUNDARY_PATH = os.path.join(PROC_DIR, 'seongnam_boundary.gpkg')

print('기업3 경로:', os.path.exists(CORP3_PATH))
corp4_files = sorted(glob.glob(CORP4_GLOB))
print(f'기업4 파일 수: {len(corp4_files)}개')
print('경계 파일:', os.path.exists(BOUNDARY_PATH))

---
## 1단계 — 데이터 전처리

In [ ]:
# ── 기업3: 202512 단일 시점 법인 현황 ──────────────────────────────────────
df3 = pd.read_csv(CORP3_PATH, encoding='utf-8-sig')

print(f'기업3 shape: {df3.shape}')
print(f'행정동 수  : {df3["admi_nm"].nunique()}개')
print(f'업종 수    : {df3["induty_pri_nm"].nunique()}개')
display(df3.head(3))

In [ ]:
# ── 기업4: 202412~202512 13개월 신설 법인 ─────────────────────────────────
frames = []
for fp in corp4_files:
    ym = os.path.basename(fp).split('_')[3]   # ex) '202412'
    tmp = pd.read_csv(fp, encoding='utf-8-sig')
    tmp['month'] = ym
    frames.append(tmp)

df4_raw = pd.concat(frames, ignore_index=True)
df4_raw['month'] = df4_raw['month'].astype(str)

print(f'기업4 전체 shape : {df4_raw.shape}')
print(f'기간             : {sorted(df4_raw["month"].unique())}')
print(f'행정동 수        : {df4_raw["admi_nm"].nunique()}개')
display(df4_raw.head(3))

In [ ]:
# 기업4 월별·행정동별 신설 법인 수 집계
df4_monthly = (
    df4_raw
    .groupby(['month', 'admi_nm'])['ncr_crp_comp_cn']
    .sum()
    .reset_index()
    .rename(columns={'ncr_crp_comp_cn': 'new_corp_cnt'})
)

print(f'월별 행정동 레코드 수: {len(df4_monthly)}')
display(df4_monthly.head(6))

---
## 2단계 — 지표 산출

### 2-1. 기업3 지표: 전체 법인 수 & 드론 친화 업종 비율

드론 배송 친화 업종 기준: **제조업** · **도매 및 소매업** · **운수 및 창고업**

In [ ]:
# 드론 친화 업종 키워드
DRONE_FRIENDLY_KEYWORDS = ['제조업', '도매 및 소매업', '운수 및 창고업']

def is_drone_friendly(industry_name: str) -> bool:
    return any(kw in str(industry_name) for kw in DRONE_FRIENDLY_KEYWORDS)

df3['is_drone_friendly'] = df3['induty_pri_nm'].apply(is_drone_friendly)

# 행정동별 전체 법인 수
total_corp = (
    df3.groupby('admi_nm')['co_ctx_comp_cn']
    .sum()
    .reset_index()
    .rename(columns={'co_ctx_comp_cn': 'total_corp'})
)

# 행정동별 드론 친화 업종 법인 수
drone_corp = (
    df3[df3['is_drone_friendly']]
    .groupby('admi_nm')['co_ctx_comp_cn']
    .sum()
    .reset_index()
    .rename(columns={'co_ctx_comp_cn': 'drone_corp'})
)

corp3_idx = total_corp.merge(drone_corp, on='admi_nm', how='left').fillna(0)
corp3_idx['drone_ratio'] = corp3_idx['drone_corp'] / corp3_idx['total_corp'].replace(0, np.nan)
corp3_idx['drone_ratio'] = corp3_idx['drone_ratio'].fillna(0)

print(f'행정동 수: {len(corp3_idx)}')
display(corp3_idx.sort_values('total_corp', ascending=False).head(10))

### 2-2. 기업4 지표: 연간 누적 신설 수 & 상·하반기 증감률

In [ ]:
# 상반기: 202412~202505 (6개월) / 하반기: 202506~202512 (7개월)
H1_MONTHS = ['202412','202501','202502','202503','202504','202505']
H2_MONTHS = ['202506','202507','202508','202509','202510','202511','202512']

# 연간 누적
annual_new = (
    df4_monthly.groupby('admi_nm')['new_corp_cnt']
    .sum()
    .reset_index()
    .rename(columns={'new_corp_cnt': 'annual_new_corp'})
)

# 상·하반기 집계
h1 = (
    df4_monthly[df4_monthly['month'].isin(H1_MONTHS)]
    .groupby('admi_nm')['new_corp_cnt'].sum()
    .reset_index().rename(columns={'new_corp_cnt': 'h1_new'})
)
h2 = (
    df4_monthly[df4_monthly['month'].isin(H2_MONTHS)]
    .groupby('admi_nm')['new_corp_cnt'].sum()
    .reset_index().rename(columns={'new_corp_cnt': 'h2_new'})
)

corp4_idx = annual_new.merge(h1, on='admi_nm', how='outer').merge(h2, on='admi_nm', how='outer').fillna(0)

# 하반기 대비 상반기 증감률 (H2/H1 - 1), H1=0이면 H2가 있으면 1.0, 없으면 0
def safe_growth_rate(row):
    if row['h1_new'] == 0:
        return 1.0 if row['h2_new'] > 0 else 0.0
    return (row['h2_new'] - row['h1_new']) / row['h1_new']

corp4_idx['hh_growth_rate'] = corp4_idx.apply(safe_growth_rate, axis=1)

# 월평균 신설 법인 수
corp4_idx['monthly_avg_new'] = corp4_idx['annual_new_corp'] / 13

print(f'행정동 수: {len(corp4_idx)}')
display(corp4_idx.sort_values('annual_new_corp', ascending=False).head(10))

In [ ]:
# 모든 행정동 기준표 생성 (기업3 50개 동 기준)
all_dongs = pd.DataFrame({'admi_nm': sorted(df3['admi_nm'].unique())})

indicators = (
    all_dongs
    .merge(corp3_idx[['admi_nm','total_corp','drone_ratio']], on='admi_nm', how='left')
    .merge(corp4_idx[['admi_nm','annual_new_corp','hh_growth_rate']], on='admi_nm', how='left')
    .fillna(0)
)

print('통합 지표 테이블:')
display(indicators.sort_values('total_corp', ascending=False).head(15))

---
## 3단계 — 기업 활동 활성화 지수 산출

| 지표 | 출처 | 가중치 |
|------|------|--------|
| 전체 법인 수 | 기업3 202512 | **0.4** |
| 드론 배송 친화 업종 비율 | 기업3 202512 | **0.3** |
| 연간 누적 신설 법인 수 | 기업4 12개월 합산 | **0.2** |
| 하반기 신설 증감률 | 기업4 추이 | **0.1** |

Min-Max 정규화 후 가중 합산

In [ ]:
def minmax(s: pd.Series) -> pd.Series:
    mn, mx = s.min(), s.max()
    if mx == mn:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - mn) / (mx - mn)

# 하반기 증감률은 극단값 클리핑 (상위 95% 캡)
cap = indicators['hh_growth_rate'].quantile(0.95)
indicators['hh_growth_rate_clipped'] = indicators['hh_growth_rate'].clip(upper=cap)

WEIGHTS = {
    'total_corp'           : 0.4,
    'drone_ratio'          : 0.3,
    'annual_new_corp'      : 0.2,
    'hh_growth_rate_clipped': 0.1,
}

for col in WEIGHTS:
    indicators[f'{col}_norm'] = minmax(indicators[col])

indicators['activity_index'] = sum(
    indicators[f'{col}_norm'] * w
    for col, w in WEIGHTS.items()
)

# 0~100 스케일로 변환
indicators['activity_index_100'] = (indicators['activity_index'] * 100).round(2)

result = indicators[[
    'admi_nm','total_corp','drone_ratio','annual_new_corp',
    'hh_growth_rate','activity_index_100'
]].sort_values('activity_index_100', ascending=False)

print('=== 기업 활동 활성화 지수 TOP 20 ===')
display(result.head(20))

---
## 4단계 — 시각화

### 4-1. 활성화 지수 순위 막대 차트

In [ ]:
TOP_N = 20
top_result = result.head(TOP_N).copy()

fig = px.bar(
    top_result.sort_values('activity_index_100'),
    x='activity_index_100',
    y='admi_nm',
    orientation='h',
    color='activity_index_100',
    color_continuous_scale='Blues',
    text='activity_index_100',
    title='행정동별 기업 활동 활성화 지수 TOP 20 (B2B 드론 배송 수요)',
    labels={
        'activity_index_100': '활성화 지수 (0~100)',
        'admi_nm': '행정동'
    }
)
fig.update_traces(texttemplate='%{text:.1f}', textposition='outside')
fig.update_layout(
    height=600,
    coloraxis_showscale=False,
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(size=12),
    title_font_size=14
)
fig.show()

### 4-2. 지표 구성 비교 — 스택 바 차트

In [ ]:
top20 = result.head(TOP_N).copy()

# 가중치 적용된 기여도
top20['기여_법인수']       = top20['total_corp_norm']            * 0.4 * 100 if 'total_corp_norm' in top20.columns else minmax(top20['total_corp']) * 0.4 * 100
top20['기여_드론친화']      = top20['drone_ratio_norm']           * 0.3 * 100 if 'drone_ratio_norm' in top20.columns else minmax(top20['drone_ratio']) * 0.3 * 100
top20['기여_신설법인']      = top20['annual_new_corp_norm']       * 0.2 * 100 if 'annual_new_corp_norm' in top20.columns else minmax(top20['annual_new_corp']) * 0.2 * 100
top20['기여_증감률']        = top20['hh_growth_rate_clipped_norm']* 0.1 * 100 if 'hh_growth_rate_clipped_norm' in top20.columns else 0

# norm 컬럼이 없으면 indicators 에서 가져오기
norm_cols = ['total_corp_norm','drone_ratio_norm','annual_new_corp_norm','hh_growth_rate_clipped_norm']
top20_merged = top20.merge(
    indicators[['admi_nm'] + norm_cols],
    on='admi_nm', how='left'
)
top20_merged['기여_법인수']  = top20_merged['total_corp_norm'] * 0.4 * 100
top20_merged['기여_드론친화'] = top20_merged['drone_ratio_norm'] * 0.3 * 100
top20_merged['기여_신설법인'] = top20_merged['annual_new_corp_norm'] * 0.2 * 100
top20_merged['기여_증감률']  = top20_merged['hh_growth_rate_clipped_norm'] * 0.1 * 100

top20_sorted = top20_merged.sort_values('activity_index_100')

fig2 = go.Figure()
colors = ['#1f77b4','#ff7f0e','#2ca02c','#d62728']
contrib_cols = ['기여_법인수','기여_드론친화','기여_신설법인','기여_증감률']
labels = ['전체 법인 수 (×0.4)','드론 친화 업종 비율 (×0.3)','연간 신설 법인 수 (×0.2)','하반기 증감률 (×0.1)']

for col, label, color in zip(contrib_cols, labels, colors):
    fig2.add_trace(go.Bar(
        name=label,
        y=top20_sorted['admi_nm'],
        x=top20_sorted[col],
        orientation='h',
        marker_color=color
    ))

fig2.update_layout(
    barmode='stack',
    title='행정동별 활성화 지수 구성 분해 (TOP 20)',
    xaxis_title='지수 기여도',
    yaxis_title='행정동',
    height=600,
    legend=dict(orientation='h', yanchor='bottom', y=1.01, xanchor='right', x=1),
    plot_bgcolor='white'
)
fig2.show()

### 4-3. 월별 신설 법인 추이 — 상위 동 라인 차트

In [ ]:
# 상위 행정동 선택 (기업4에 데이터가 있는 동 중 활성화 지수 상위)
top_dongs_with_data = [
    d for d in result['admi_nm'].tolist()
    if d in df4_monthly['admi_nm'].unique()
][:8]

df4_top = df4_monthly[df4_monthly['admi_nm'].isin(top_dongs_with_data)].copy()

# month를 날짜 형식으로 변환
df4_top['date'] = pd.to_datetime(df4_top['month'], format='%Y%m')
df4_top = df4_top.sort_values('date')

# 전체 월 × 동 조합 완성 (결측 → 0)
all_months = sorted(df4_monthly['month'].unique())
full_idx = pd.MultiIndex.from_product(
    [all_months, top_dongs_with_data], names=['month','admi_nm']
)
df4_full = (
    df4_top.set_index(['month','admi_nm'])['new_corp_cnt']
    .reindex(full_idx, fill_value=0)
    .reset_index()
)
df4_full['date'] = pd.to_datetime(df4_full['month'], format='%Y%m')
df4_full['month_label'] = df4_full['date'].dt.strftime('%y.%m')

fig3 = px.line(
    df4_full,
    x='month_label',
    y='new_corp_cnt',
    color='admi_nm',
    markers=True,
    title='행정동별 월별 신설 법인 수 추이 (202412~202512)',
    labels={
        'month_label': '연월',
        'new_corp_cnt': '신설 법인 수',
        'admi_nm': '행정동'
    }
)

# 상·하반기 구분선
fig3.add_vline(
    x='25.06', line_dash='dash', line_color='red', opacity=0.6,
    annotation_text='하반기 시작', annotation_position='top right'
)

fig3.update_layout(
    height=500,
    plot_bgcolor='white',
    legend=dict(orientation='h', yanchor='bottom', y=1.01, xanchor='right', x=1)
)
fig3.show()

In [ ]:
# 추이 패턴 분류: 꾸준한 증가 vs 최근 급증
print('=== 상/하반기 신설 법인 증감률 ===\n')

pattern_df = corp4_idx[corp4_idx['admi_nm'].isin(top_dongs_with_data)].copy()
pattern_df['패턴'] = pattern_df['hh_growth_rate'].apply(
    lambda r: '최근 급증 (하반기 가속)' if r > 0.3
    else ('꾸준한 증가' if r >= 0 else '상반기 집중')
)

display(
    pattern_df[['admi_nm','h1_new','h2_new','hh_growth_rate','패턴']]
    .sort_values('hh_growth_rate', ascending=False)
    .rename(columns={
        'admi_nm':'행정동',
        'h1_new':'상반기 신설',
        'h2_new':'하반기 신설',
        'hh_growth_rate':'증감률'
    })
)

### 4-4. 단계구분도 (Choropleth Map)

In [ ]:
# 행정 경계 GeoPackage 로드
layers = gpd.list_layers(BOUNDARY_PATH)
print('사용 가능한 레이어:', layers)

In [ ]:
# 레이어 로드 (첫 번째 레이어 자동 사용)
gdf = gpd.read_file(BOUNDARY_PATH, layer=layers.iloc[0]['name'])
print(f'GeoDataFrame shape: {gdf.shape}')
print('컬럼:', gdf.columns.tolist())
gdf.head(3)

In [ ]:
# 행정동 이름 컬럼 자동 탐색
dong_col = None
for col in gdf.columns:
    sample = gdf[col].dropna().astype(str).iloc[:5].tolist()
    if any('동' in s or '구' in s for s in sample):
        # 기업3 행정동과 교집합 확인
        overlap = len(set(gdf[col].astype(str)) & set(indicators['admi_nm']))
        if overlap > 5:
            dong_col = col
            print(f'행정동 컬럼 발견: "{col}" (교집합 {overlap}개)')
            break

if dong_col is None:
    print('컬럼 샘플:'); print(gdf.iloc[:3].to_string())
    dong_col = input('행정동 이름 컬럼명을 직접 입력하세요: ').strip()

In [ ]:
# GeoDataFrame에 활성화 지수 합류
gdf_merged = gdf.merge(
    indicators[['admi_nm','activity_index_100','total_corp','drone_ratio','annual_new_corp','hh_growth_rate']],
    left_on=dong_col, right_on='admi_nm', how='left'
)

# WGS84로 변환 (folium 필요)
if gdf_merged.crs and gdf_merged.crs.to_epsg() != 4326:
    gdf_merged = gdf_merged.to_crs(epsg=4326)

print(f'매칭 성공: {gdf_merged["activity_index_100"].notna().sum()} / {len(gdf_merged)}')
print(f'매칭 실패 행정동: {gdf_merged[gdf_merged["activity_index_100"].isna()][dong_col].tolist()}')

In [ ]:
# Folium 단계구분도
center = [gdf_merged.geometry.centroid.y.mean(), gdf_merged.geometry.centroid.x.mean()]

m = folium.Map(location=center, zoom_start=12, tiles='CartoDB positron')

# GeoJSON 변환
geojson_data = json.loads(gdf_merged.to_json())

Choropleth(
    geo_data=geojson_data,
    data=gdf_merged,
    columns=[dong_col, 'activity_index_100'],
    key_on=f'feature.properties.{dong_col}',
    fill_color='YlOrRd',
    fill_opacity=0.75,
    line_opacity=0.5,
    legend_name='기업 활동 활성화 지수 (0~100)',
    nan_fill_color='#cccccc',
    nan_fill_opacity=0.4
).add_to(m)

# 툴팁 추가
folium.GeoJson(
    geojson_data,
    tooltip=folium.GeoJsonTooltip(
        fields=[dong_col, 'activity_index_100', 'total_corp', 'annual_new_corp'],
        aliases=['행정동', '활성화 지수', '전체 법인 수', '연간 신설 법인'],
        localize=True,
        sticky=True
    ),
    style_function=lambda x: {'fillOpacity': 0, 'weight': 0}
).add_to(m)

# 상위 10개 동 마커
top10 = result.head(10)
for _, row in gdf_merged[gdf_merged['admi_nm'].isin(top10['admi_nm'])].iterrows():
    centroid = row.geometry.centroid
    folium.Marker(
        location=[centroid.y, centroid.x],
        popup=folium.Popup(
            f"<b>{row[dong_col]}</b><br>지수: {row['activity_index_100']:.1f}<br>"
            f"법인 수: {int(row['total_corp'])}개<br>연간 신설: {int(row['annual_new_corp'] if pd.notna(row['annual_new_corp']) else 0)}개",
            max_width=200
        ),
        icon=folium.Icon(color='red', icon='building', prefix='fa')
    ).add_to(m)

folium.LayerControl().add_to(m)

# HTML 저장
map_path = os.path.join(PROC_DIR, 'b2b_activity_index_map.html')
m.save(map_path)
print(f'지도 저장: {map_path}')
m

### 4-5. 전체 지표 히트맵

In [ ]:
import seaborn as sns

heatmap_cols = {
    'total_corp_norm': '전체 법인 수',
    'drone_ratio_norm': '드론 친화 비율',
    'annual_new_corp_norm': '연간 신설 법인',
    'hh_growth_rate_clipped_norm': '하반기 증감률'
}

hm_data = (
    indicators
    .sort_values('activity_index_100', ascending=False)
    .head(20)
    .set_index('admi_nm')[list(heatmap_cols.keys())]
    .rename(columns=heatmap_cols)
)

fig_hm, ax = plt.subplots(figsize=(8, 8))
sns.heatmap(
    hm_data,
    cmap='YlOrRd',
    linewidths=0.5,
    annot=True,
    fmt='.2f',
    ax=ax,
    cbar_kws={'label': '정규화 지수 (0~1)'}
)
ax.set_title('행정동별 B2B 드론 수요 지표 히트맵 (TOP 20)', fontsize=13, pad=12)
ax.set_xlabel('')
ax.set_ylabel('')
plt.tight_layout()
hm_path = os.path.join(PROC_DIR, 'b2b_heatmap.png')
plt.savefig(hm_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'히트맵 저장: {hm_path}')

---
## 결과 저장

In [ ]:
# 최종 결과 CSV 저장
out_path = os.path.join(PROC_DIR, 'b2b_activity_index.csv')
result.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f'저장 완료: {out_path}')

# 월별 신설 법인 집계 저장
monthly_out = os.path.join(PROC_DIR, 'b2b_monthly_new_corp.csv')
df4_monthly.to_csv(monthly_out, index=False, encoding='utf-8-sig')
print(f'저장 완료: {monthly_out}')

print('\n=== 최종 결과 요약 ===')
print(f'분석 대상 행정동 수  : {len(result)}개')
print(f'TOP 5 버티포트 후보 :')
for i, (_, row) in enumerate(result.head(5).iterrows(), 1):
    print(f'  {i}위 {row["admi_nm"]:8s} | 지수 {row["activity_index_100"]:5.1f} | '
          f'법인 {int(row["total_corp"]):4d}개 | 신설 {int(row["annual_new_corp"]):3d}개')

---
## 한계 및 유의사항

1. **기업3 단일 시점 한계** — 202512 단일 스냅샷으로 법인 밀집도 변화 추이를 확인할 수 없습니다. 과거 시점 데이터 추가 시 트렌드 분석이 가능합니다.

2. **기업4 1년치 계절성** — 13개월 데이터는 계절성 패턴 파악에는 충분하나 3년 이상의 장기 트렌드 해석에는 제한적입니다.

3. **업종 코드 불일치** — 기업3과 기업4의 업종 코드 체계가 상이합니다(기업3: 업종명 직접 기재, 기업4: 알파벳 접두어 포함). 드론 친화 업종은 기업3 기준으로만 산출되었습니다.

4. **행정동 범위 차이** — 기업4에는 신설 법인이 없는 행정동이 집계에서 제외됩니다. 해당 동의 `annual_new_corp`는 0으로 처리하였습니다.

5. **B2B 수요 프록시** — 법인 수와 신설 추이는 실제 B2B 배송 수요의 간접 지표입니다. 실제 물동량 데이터와 결합 시 정확도가 향상됩니다.